In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
train = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/train.parquet')
test = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/test.parquet')
train_input = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/train_input.parquet')
val_input = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/val_input.parquet')
val = pd.read_parquet('/kaggle/input/train-test-val-parquet-final/val.parquet')

train_target = train.loc[train_input.index]['fare_amount']
val_target = val.loc[val_input.index]['fare_amount']

In [ ]:
from  sklearn.tree import DecisionTreeRegressor

def max_depth_test(x):
    model_dt = DecisionTreeRegressor(max_depth = x, random_state=123)
    model_dt.fit(train_input, train_target)
    train_pred = model_dt.predict(train_input)
    val_pred = model_dt.predict(val_input)
    train_rmse = np.sqrt(mean_squared_error(train_target,train_pred))
    val_rmse = np.sqrt(mean_squared_error(val_target,val_pred))
    return {'max_depth': x,  'training error': train_rmse, 'validation error': val_rmse}

errors_max_depth = pd.DataFrame([max_depth_test(md) for md in range (5,51,5)])

def min_samples_split_test(x):
    model_dt = DecisionTreeRegressor(min_samples_split = x, random_state=123)
    model_dt.fit(train_input, train_target)
    train_pred = model_dt.predict(train_input)
    val_pred = model_dt.predict(val_input)
    train_rmse = np.sqrt(mean_squared_error(train_target,train_pred))
    val_rmse = np.sqrt(mean_squared_error(val_target,val_pred))
    return {'min_samples_split': x,  'training error': train_rmse, 'validation error': val_rmse}

errors_min_samples_split = pd.DataFrame([min_samples_split_test(md) for md in range (5,101,5)])

def min_samples_leaf_test(x):
    model_dt = DecisionTreeRegressor(min_samples_leaf = x, random_state=123)
    model_dt.fit(train_input, train_target)
    train_pred = model_dt.predict(train_input)
    val_pred = model_dt.predict(val_input)
    train_rmse = np.sqrt(mean_squared_error(train_target,train_pred))
    val_rmse = np.sqrt(mean_squared_error(val_target,val_pred))
    return {'min_samples_leaf': x,  'training error': train_rmse, 'validation error': val_rmse}

errors_min_samples_split = pd.DataFrame([min_samples_leaf_test(md) for md in range (5,101,5)])

def final_param_test(x):
    model_dt = x
    model_dt.fit(train_input, train_target)
    train_pred = model_dt.predict(train_input)
    val_pred = model_dt.predict(val_input)
    train_rmse = np.sqrt(mean_squared_error(train_target,train_pred))
    val_rmse = np.sqrt(mean_squared_error(val_target,val_pred))
    return {'training error': train_rmse, 'validation error': val_rmse}

model_dt1 = DecisionTreeRegressor(min_samples_split = 75,max_depth = 10, min_samples_leaf = 30, splitter = 'best', criterion = 'squared_error')
model_dt2 = DecisionTreeRegressor(min_samples_split = 75,max_depth = 10, min_samples_leaf = 30, splitter = 'random', criterion = 'squared_error')

best_sample_splitter = pd.DataFrame([final_param_test(model_dt1)])
best_sample_splitter2 = pd.DataFrame([final_param_test(model_dt1)])

# when submitted to kaggele final score was 3.72673 so 831 on leaderboard
model_dt = DecisionTreeRegressor(max_depth =15 , min_samples_leaf = 35, random_state=123)
model_dt.fit(train_input, train_target)
train_pred = model_dt.predict(train_input)
val_pred = model_dt.predict(val_input)
train_rmse = np.sqrt(mean_squared_error(train_target,train_pred))
val_rmse = np.sqrt(mean_squared_error(val_target,val_pred))

test_pred_dt = model_dt.predict(test)
submission_dt = pd.read_csv('/kaggle/input/nyc-taxi-fare-datasets/sample_submission.csv')
submission_dt['fare_amount'] = test_pred_dt
submission_dt.to_csv('decision_tree.csv', index=None)

#model trained on 50% of the data. submission score is 3.65559
model_dt2 = DecisionTreeRegressor(max_depth =15 , min_samples_leaf = 35, random_state=123)
model_dt2.fit(train2_input, train2_target)
train2_pred = model_dt2.predict(train2_input)
val2_pred = model_dt.predict(val2_input)
train2_rmse = np.sqrt(mean_squared_error(train2_target,train2_pred))
val2_rmse = np.sqrt(mean_squared_error(val2_target,val2_pred))
print(f'trian2_rmse: {train2_rmse}, val2_rmse: {val2_rmse}')

test_pred_dt2 = model_dt2.predict(test)
submission_dt2 = pd.read_csv('/kaggle/input/nyc-taxi-fare-datasets/sample_submission.csv')
submission_dt2['fare_amount'] = test_pred_dt2
submission_dt2.to_csv('decision_tree_sub2.csv', index=None)